### SMOTE (BEST)

In [3]:

import h5py
import numpy as np
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, cohen_kappa_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

# -------------------- Custom scoring for Wake --------------------
wake_recall = make_scorer(recall_score, pos_label=1)

# -------------------- Load data --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train_old = f['X_train'][:]
    X_test_old = f['X_test'][:]
    y_train_old = f['y_train'][:]
    y_test_old = f['y_test'][:]

# -------------------- New 80/20 split --------------------
X_all = np.concatenate([X_train_old, X_test_old], axis=0)
y_all = np.concatenate([y_train_old, y_test_old], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

# -------------------- Binary mapping --------------------
sleep_stages = [1, 2, 3, 5]
y_train_bin = np.where(np.isin(y_train, sleep_stages), 0, 1)
y_test_bin  = np.where(np.isin(y_test, sleep_stages), 0, 1)

print("Before SMOTE:", Counter(y_train_bin))

# -------------------- SMOTE (moderate) --------------------
smote = SMOTE(
    sampling_strategy=0.6,   # Wake = 50% of Sleep
    k_neighbors=3,
    random_state=42
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train_bin)
print("After SMOTE:", Counter(y_train_sm))

# -------------------- AdaBoost base tree --------------------
base_tree = DecisionTreeClassifier(random_state=42)

ada = AdaBoostClassifier(
    estimator=base_tree,
    random_state=42
)

# -------------------- Hyperparameter grid --------------------
param_grid = {
    "estimator__max_depth": [5],
    "estimator__min_samples_leaf": [10],
    "n_estimators": [50],
    "learning_rate": [0.1]
}

# -------------------- Grid Search --------------------
grid = GridSearchCV(
    ada,
    param_grid,
    scoring=wake_recall,   # focus on Wake
    cv=2,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train_sm, y_train_sm)

print("\nBest parameters:", grid.best_params_)
print("Best Wake recall (CV):", grid.best_score_)

best_model = grid.best_estimator_
# -------------------- Prediction --------------------
y_pred = best_model.predict(X_test)

# -------------------- Metrics --------------------
accuracy = accuracy_score(y_test_bin, y_pred)
precision = precision_score(y_test_bin, y_pred, zero_division=0)
recall = recall_score(y_test_bin, y_pred, zero_division=0)
f1 = f1_score(y_test_bin, y_pred, zero_division=0)
kappa = cohen_kappa_score(y_test_bin, y_pred)

cm = confusion_matrix(y_test_bin, y_pred)
sleep_recall = cm[0,0] / (cm[0,0] + cm[0,1])
wake_recall  = cm[1,1] / (cm[1,1] + cm[1,0])

print("\nEvaluation Metrics:")
print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision (Wake):      {precision:.4f}")
print(f"Recall (Wake):         {recall:.4f}")
print(f"Sleep Recall:          {sleep_recall:.4f}")
print(f"Wake Recall:           {wake_recall:.4f}")
print(f"F1-score:              {f1:.4f}")
print(f"Cohen's Kappa:         {kappa:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_bin, y_pred,
    target_names=["Sleep", "Wake"],
    zero_division=0
))

Before SMOTE: Counter({np.int64(0): 38831, np.int64(1): 7567})
After SMOTE: Counter({np.int64(0): 38831, np.int64(1): 23298})
Fitting 2 folds for each of 1 candidates, totalling 2 fits

Best parameters: {'estimator__max_depth': 5, 'estimator__min_samples_leaf': 10, 'learning_rate': 0.1, 'n_estimators': 50}
Best Wake recall (CV): 0.9133402008756116

Evaluation Metrics:
Accuracy:              0.9332
Precision (Wake):      0.7531
Recall (Wake):         0.8784
Sleep Recall:          0.9439
Wake Recall:           0.8784
F1-score:              0.8109
Cohen's Kappa:         0.7706

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.98      0.94      0.96      9708
        Wake       0.75      0.88      0.81      1892

    accuracy                           0.93     11600
   macro avg       0.86      0.91      0.89     11600
weighted avg       0.94      0.93      0.94     11600



# Test 1

In [1]:
import h5py
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.utils import resample
import time

# -------------------- Sensitivity & Specificity --------------------
def sensitivity_score(y_true, y_pred):
    """Sensitivity = Recall for Sleep (positive class = 0)."""
    cm = confusion_matrix(y_true, y_pred)
    tp = cm[0, 0]
    fn = cm[0, 1]
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def specificity_score(y_true, y_pred):
    """Specificity = Recall for Wake (negative class = 1)."""
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[1, 1]
    fp = cm[1, 0]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- Binary AdaBoost classifier --------------------
def adaboost_classifier(X_train, y_train, X_test, y_test, use_scaling=True, n_estimators=100, learning_rate=1.0, max_depth=1):
    print("\n--- AdaBoost for Sleep vs Awake classification ---")
    
    # Scale features
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Base estimator
    base_estimator = DecisionTreeClassifier(max_depth=max_depth, class_weight='balanced', random_state=42)
    
    # Initialize AdaBoost
    clf = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=42
    )
    
    # Fit AdaBoost
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"AdaBoost fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)  # recall for Wake (label=1)
    sensitivity = sensitivity_score(y_test, y_pred)          # Sleep
    specificity = specificity_score(y_test, y_pred)          # Wake
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall (Wake):  {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:       {f1:.4f}")
    print(f"Cohen Kappa:    {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- RandomizedSearchCV for AdaBoost --------------------
def tune_adaboost(X_train, y_train):
    print("\n🔍 Running Random Search for AdaBoost Hyperparameter Tuning (Binary)...")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    base_estimator = DecisionTreeClassifier(class_weight='balanced', random_state=42)
    ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)
    
    param_dist = {
    "n_estimators": [50, 100],         # fewer choices
    "learning_rate": [0.1, 0.5, 1.0], # small set
    "estimator__max_depth": [1, 2]    # shallow trees
        }
   

    
    search = RandomizedSearchCV(
        ada,
        param_distributions=param_dist,
        n_iter=5,  # smaller number for faster tuning
        cv=2,
        scoring="f1_macro",
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load HDF5 train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# --- New 80/20 split ---
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)

sleep_stages = [1,2,3,5]
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0

y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

print("Unique train labels:", np.unique(y_train_bin, return_counts=True))
print("Unique test labels:", np.unique(y_test_bin, return_counts=True))

# -------------------- Upsample Wake --------------------
X_train_sleep = X_train[y_train_bin == 0]
y_train_sleep = y_train_bin[y_train_bin == 0]
X_train_wake = X_train[y_train_bin == 1]
y_train_wake = y_train_bin[y_train_bin == 1]

if len(X_train_sleep) > len(X_train_wake):
    X_wake_upsampled, y_wake_upsampled = resample(
        X_train_wake, y_train_wake,
        replace=True,
        n_samples=len(X_train_sleep),
        random_state=42
    )
    X_train_balanced = np.vstack([X_train_sleep, X_wake_upsampled])
    y_train_balanced = np.hstack([y_train_sleep, y_wake_upsampled])
else:
    X_train_balanced = np.vstack([X_train_sleep, X_train_wake])
    y_train_balanced = np.hstack([y_train_sleep, y_train_wake])

print(f"Balanced training set: {X_train_balanced.shape[0]} samples")

# -------------------- Tune AdaBoost --------------------
best_params_ada = tune_adaboost(X_train_balanced, y_train_balanced)

# -------------------- Train final AdaBoost --------------------
ada_binary = adaboost_classifier(
    X_train_balanced, y_train_balanced,
    X_test, y_test_bin,
    use_scaling=True,
    n_estimators=best_params_ada['n_estimators'],
    learning_rate=best_params_ada['learning_rate'],
    max_depth=best_params_ada['estimator__max_depth']
)


Training samples: 46398, Test samples: 11600
Unique train labels: (array([0, 1]), array([38831,  7485]))
Unique test labels: (array([0, 1]), array([9708, 1872]))
Balanced training set: 77662 samples

🔍 Running Random Search for AdaBoost Hyperparameter Tuning (Binary)...
Fitting 2 folds for each of 5 candidates, totalling 10 fits

✅ Best hyperparameters: {'n_estimators': 100, 'learning_rate': 0.5, 'estimator__max_depth': 2}
🏅 Best CV score (F1 macro): 0.9188

--- AdaBoost for Sleep vs Awake classification ---
Data scaled with StandardScaler.
AdaBoost fitted in 213.00 seconds.

Evaluation Metrics:
Accuracy:       0.9129
Precision:      0.6691
Recall (Wake):  0.9119
Sensitivity (Sleep): 0.9131
Specificity (Wake):  0.9119
F1 Score:       0.7719
Cohen Kappa:    0.7196

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.98      0.91      0.95      9708
        Wake       0.67      0.91      0.77      1872

    accuracy                          

### Adasyn

In [3]:
import h5py
import numpy as np
import time

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    cohen_kappa_score
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from imblearn.over_sampling import ADASYN
from collections import Counter

def sensitivity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tp = cm[0, 0]
    fn = cm[0, 1]
    return tp / (tp + fn) if (tp + fn) else 0


def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[1, 1]
    fp = cm[1, 0]
    return tn / (tn + fp) if (tn + fp) else 0

def adaboost_classifier(
    X_train, y_train,
    X_test, y_test,
    use_scaling=True,
    n_estimators=50,
    learning_rate=1.0,
    max_depth=1
):
    print("\n--- AdaBoost + ADASYN: Sleep vs Wake ---")

    if use_scaling:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    base_estimator = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=42
    )

    clf = AdaBoostClassifier(
        estimator=base_estimator,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=42
    )

    start = time.time()
    clf.fit(X_train, y_train)
    print(f"AdaBoost trained in {time.time() - start:.2f}s")

    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    sensitivity = sensitivity_score(y_test, y_pred)
    specificity = specificity_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    kappa = cohen_kappa_score(y_test, y_pred)

    print("\nEvaluation Metrics:")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (Wake):    {precision:.4f}")
    print(f"Recall (Wake):       {recall:.4f}")
    print(f"Sensitivity (Sleep): {sensitivity:.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:            {f1:.4f}")
    print(f"Cohen’s Kappa:       {kappa:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Sleep", "Wake"],
        zero_division=0
    ))

    return clf


def tune_adaboost(X_train, y_train):
    print("\n🔍 Hyperparameter tuning (AdaBoost + ADASYN)")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    base_estimator = DecisionTreeClassifier(random_state=42)

    ada = AdaBoostClassifier(
        estimator=base_estimator,
        random_state=42
    )

    param_dist = {
        "n_estimators": [50, 100],
        "learning_rate": [0.1, 0.5, 1.0],
        "estimator__max_depth": [1, 2]
    }

    search = RandomizedSearchCV(
        ada,
        param_distributions=param_dist,
        n_iter=5,
        cv=2,
        scoring="f1_macro",
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, y_train)
    print("Best params:", search.best_params_)

    return search.best_params_

hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, "r") as f:
    X_train_old = f["X_train"][:]
    X_test_old  = f["X_test"][:]
    y_train_old = f["y_train"][:]
    y_test_old  = f["y_test"][:]

X_all = np.concatenate([X_train_old, X_test_old])
y_all = np.concatenate([y_train_old, y_test_old])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)

valid_labels = [0, 1, 2, 3, 5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]

y_train_bin = np.zeros_like(y_train)
y_test_bin  = np.zeros_like(y_test)

y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

print("Train distribution:", np.unique(y_train_bin, return_counts=True))
print("Test distribution :", np.unique(y_test_bin, return_counts=True))
print("\nApplying ADASYN...")
adasyn = ADASYN(random_state=42)
X_train_os, y_train_os = adasyn.fit_resample(X_train, y_train_bin)

print("After ADASYN:", Counter(y_train_os))


best_params = tune_adaboost(X_train_os, y_train_os)

ada_model = adaboost_classifier(
    X_train_os, y_train_os,
    X_test, y_test_bin,
    use_scaling=True,
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    max_depth=best_params["estimator__max_depth"]
)


Train distribution: (array([0, 1]), array([38831,  7485]))
Test distribution : (array([0, 1]), array([9708, 1872]))

Applying ADASYN...
After ADASYN: Counter({np.int64(1): 39264, np.int64(0): 38831})

🔍 Hyperparameter tuning (AdaBoost + ADASYN)
Fitting 2 folds for each of 5 candidates, totalling 10 fits
Best params: {'n_estimators': 100, 'learning_rate': 0.5, 'estimator__max_depth': 2}

--- AdaBoost + ADASYN: Sleep vs Wake ---
AdaBoost trained in 279.85s

Evaluation Metrics:
Accuracy:            0.8960
Precision (Wake):    0.6192
Recall (Wake):       0.9268
Sensitivity (Sleep): 0.8901
Specificity (Wake):  0.9268
F1 Score:            0.7424
Cohen’s Kappa:       0.6805

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.98      0.89      0.93      9708
        Wake       0.62      0.93      0.74      1872

    accuracy                           0.90     11580
   macro avg       0.80      0.91      0.84     11580
weighted avg       0.93     